In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd
import pickle
from tqdm import tqdm

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# function name
str_function_name = 'genxii-parse-payloads'

Project: 20231010-gen-xii


### Functions

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### 1. Create container

### Create ```Dockerfile```

In [4]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy functions.py
COPY functions.py ${LAMBDA_TASK_ROOT}

# copy api.py
COPY api.py ${LAMBDA_TASK_ROOT}

# copy parser
COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}

# copy preprocessing
COPY preprocessing.py ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [5]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
scikit_learn==0.24.1
boto3==1.24.59
catboost==1.0.4

Writing requirements.txt


### Copy files

In [6]:
list_str_filename = [
    'api.py',
    'cls_parser.pkl',
    'functions.py',
    'preprocessing.py',
]
for str_filename in list_str_filename:
    str_source = f'../../07_api/01_flask_app/app/{str_filename}'
    str_destination = f'./{str_filename}'
    # copy
    shutil.copyfile(str_source, str_destination)

### Make sure all TU features are included

In [7]:
list_cols_raw_all = []
for str_model in tqdm(['01_ad','02_pricing_pd','03_pricing_lgd']):
    str_filename = 'df_test_noleaks.gzip'
    str_uri = f's3://{str_project}/{str_model}/01_data_prep/05_leaky_features/04_write_dfs/{str_filename}'
    list_cols_raw = list(pd.read_parquet(str_uri).columns)
    list_cols_raw_all.extend(list_cols_raw)
# rm dups
list_cols_raw_all = list(dict.fromkeys(list_cols_raw_all))
# rm base
list_cols_raw_all = [col for col in list_cols_raw_all if 'base' not in col]

100%|██████████| 3/3 [00:08<00:00,  2.85s/it]


### Load parser

In [8]:
str_filename = 'cls_parser.pkl'
str_local_path = f'./{str_filename}'
cls_parser = pickle.load(open(str_local_path, 'rb'))

### Assign and re-pickle

In [9]:
cls_parser.list_cols_raw = list_cols_raw_all
# re-pickle
str_filename = 'cls_parser.pkl'
str_local_path = f'./{str_filename}'
pickle.dump(cls_parser, open(str_local_path, 'wb'))

### Write ```lambda_function.py```

In [10]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
from datetime import datetime
import pickle
import json

# lambda handler
def lambda_handler(event, context):
    # get the input
    try:
        int_rows_to_parse = int(event['row'])
    except:
        int_rows_to_parse = 1
    print(f'Parsing rows: {int_rows_to_parse}')
    
    # constants
    str_project = '20231010-gen-xii'
    str_task = '13_payload_parsing'
    str_subtask = 'parsed_payloads'
    
    # get today's date
    str_date_today = datetime.today().strftime('%Y%m%d')
    
    # load requests
    print('Loading requests...')
    str_filename = f'df_rows_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/{str_task}/days/{str_date_today}/split_payloads/{str_filename}'
    df = pd.read_parquet(str_uri)
    print(f'There are {df.shape[0]} requests for this lambda function to parse')
    
    # load parser
    print('Loading parser...')
    str_filename = 'cls_parser.pkl'
    str_local_path = f'./{str_filename}'
    cls_parser = pickle.load(open(str_local_path, 'rb'))
    
    # parse (get income, LN, and TU)
    print('Parsing requests...')
    list_X_raw = []
    for a, str_request in enumerate(df['REQUEST_JSON']):
        # get bigAccountId
        int_bigaccountid = df['ACCOUNTID'].iloc[a]
        # get applicationdate
        dtm_app_date = df['REQUEST_DATETIME'].iloc[a]
    
        # convert string request to dict
        dict_json_request = json.loads(str_request)
        # get data
        cls_parser.get_data(dict_json_request)
        # parse data
        cls_parser.parse_data()
        # create X
        cls_parser.create_x()
        # get X_raw
        X_raw = cls_parser.dict_output['X_raw']
        
        # assign
        X_raw['BIGACCOUNTID'] = int_bigaccountid
        X_raw['DTM_APP_DATE'] = dtm_app_date
        # get nrows
        int_nrows = X_raw.shape[0]
        if int_nrows == 1:
            X_raw['BITDEBTOR'] = 1
        else:
            X_raw['BITDEBTOR'] = [1,0]
        
        # append
        list_X_raw.append(X_raw)
    
    # concat
    print('Concatenating raw data...')
    X_raw = pd.concat(list_X_raw)
    
    # save memeory
    del list_X_raw
    
    # write to s3 as parquet
    print('Writing raw data to s3...')
    # set nonnumeric to string
    for col in X_raw.columns:
        # if not numeric
        if X_raw[col].dtype not in ['int64', 'float64']:
            # set as string
            X_raw[col] = X_raw[col].astype(str)
        else:
            pass
    # write to gzip
    str_filename = f'X_raw_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/{str_task}/days/{str_date_today}/{str_subtask}/{str_filename}'
    X_raw.to_parquet(str_uri, compression='gzip')

Writing lambda_function.py


### Build image and push to ECR

In [11]:
%%sh

# name the image
image=genxii-parse-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon    4.4MB
Step 1/10 : FROM public.ecr.aws/lambda/python:3.8
 ---> ba8ed0cdb5a3
Step 2/10 : RUN pip install --upgrade pip
 ---> Using cache
 ---> be6f068b9c1e
Step 3/10 : COPY requirements.txt  .
 ---> Using cache
 ---> d7691bcf6c04
Step 4/10 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> ae55abe5ad00
Step 5/10 : COPY functions.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> 8e7686bfee94
Step 6/10 : COPY api.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> ced8f9aebd04
Step 7/10 : COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}
 ---> eb60c45bd7b3
Step 8/10 : COPY preprocessing.py ${LAMBDA_TASK_ROOT}
 ---> be4128d2fba7
Step 9/10 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 80ecd5c4a379
Step 10/10 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 028c699078ad
Removing intermediate container 028c699078ad
 ---> e0393e998f29
Successfully built e0393e998f29
Successfully tagged genxii-parse-payloads:lates

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-parse-payloads' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-parse-payloads]
7cba6108acd7: Preparing
fbd1bb0032e3: Preparing
8cda9699856f: Preparing
d1b0a2182ce9: Preparing
d9c05ff42b6f: Preparing
ddd56883c7fb: Preparing
946153f9fbce: Preparing
5dc5e4b28d0e: Preparing
61728103c778: Preparing
ddd56883c7fb: Waiting
946153f9fbce: Waiting
5dc5e4b28d0e: Waiting
11c57b9ec508: Preparing
59e530c8b37a: Preparing
eb4ec3a6ddd2: Preparing
71786097e499: Preparing
25a97811023e: Preparing
11c57b9ec508: Waiting
59e530c8b37a: Waiting
eb4ec3a6ddd2: Waiting
71786097e499: Waiting
25a97811023e: Waiting
61728103c778: Waiting
d9c05ff42b6f: Layer already exists
d1b0a2182ce9: Layer already exists
ddd56883c7fb: Layer already exists
946153f9fbce: Layer already exists
5dc5e4b28d0e: Layer already exists
61728103c778: Layer already exists
59e530c8b37a: Layer already exists
11c57b9ec508: Layer already exists
eb4ec3a6ddd2: Layer already exists
71786097e499: Layer already exists
25a97811023e: Lay

### 2. Create lambda function from image

In [12]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [13]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [14]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 27 Mar 2024 13:46:24 GMT',
                                      'x-amzn-requestid': '8bf43e1d-6fb8-45fb-91fa-e0dd42bd0359'},
                      'HTTPStatusCode': 204,
                      'RequestId': '8bf43e1d-6fb8-45fb-91fa-e0dd42bd0359',
                      'RetryAttempts': 0}}


In [15]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900, # 15 minutes is maximum
    MemorySize=1000, # 1000 mb == 1 gb
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000, # 1000 mb == 1 gb
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '8bc8fee4391623d4e2a97dd71e2768b8a4427313c66509f837a97f52dd4b79eb',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-parse-payloads',
 'FunctionName': 'genxii-parse-payloads',
 'LastModified': '2024-03-27T13:46:24.085+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-parse-payloads'},
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1198',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 27 Mar 2024 13:46:24 GMT',
                                      'x-amzn-requestid': '61903b2d-aeb1-47f5-8df2-dc569d2d19e4'},
                      'HTTPStatusCode': 201,
                      'RequestId': '61903b2d-aeb1-47f

### Clean-up

In [16]:
list_str_filename = list_str_filename + ['Dockerfile', 'lambda_function.py', 'requirements.txt']

for str_file in list_str_filename:
    os.remove(str_file)